---
# RL Basics 0: Problem Formulation
---

This first notebook sets up the **vocabulary** of reinforcement learning: what an
*environment* is, what a *policy* is, how the two interact to produce a *trajectory*,
and what quantity we are ultimately trying to maximise.

We work throughout with **tabular** RL, where the environment admits a **finite** number
of **states** and the agent chooses among a **finite** set of **actions** in each of them.

<center>
<img src="../src/rl_basics/imgs/minecraft_interaction_loop.png" width=80% />

[*Original Image Source*](https://medium.com/student-technical-community-vit-vellore/a-brief-introduction-to-reinforcement-learning-6a74f5a61834)
</center>

No learning happens here. We build the pieces, then hand-write a policy and measure how
good it is. Everything that follows — planning with a known model in `RL01`, learning
from samples in `RL02` — is expressed in the terms defined here.

In this notebook you will find some **⭐ Exercise**s where you need to implement missing
parts in the code. When you need to complete some code, the section is marked as:
```python
# Your code goes here: -------------------------------------

# ----------------------------------------------------------
```

# ⚙️ Setup

Run the cell below to install and import all the notebook requirements.

_Remark._ If you're running the notebook on your machine, the environment is managed with `uv` (Python 3.12+). Run `uv sync` in the project root, then select the `.venv` kernel.

In [2]:
%load_ext autoreload
%autoreload 2

from rl_basics.utils import *
setup()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/home/jhbf/Documents/U Rosario/rl/reinforcement-learning/activities/rl-basics/.venv/lib/python3.13/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment CliffWalking-RLSS-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


**Let's begin!**

# 🧩 Problem Formulation

## Environment

**Markov Decision Process.** The fundamental model behind a RL problem is the **Markov Decision Process** (MDP), defined by:
* A finite set of **states**: $\mathcal{S}$, $|\mathcal{S}| < +\infty$.
* A subset of **terminal states**: $\mathcal{S}^- \subseteq \mathcal{S}$.
* A finite set of **actions**: $\mathcal{A}$, $|\mathcal{A}| < +\infty$.
* A state **transition** probability **matrix**: $P(\cdot \mid s, a) \in \Delta(\mathcal{S})$.
* A **reward function**: $R(s, a)$.
* An **initial state distribution**: $\mu_0 \in \Delta(\mathcal{S})$.

_Remark._ The transition matrix and the initial state distribution are unknown: this is where learning comes into play!

**Gymnasium.** The `gymnasium` Python library models an MDP through the `Env` (_environment_) interface.

```python
class Env:
    def step(action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        ...
    
    def reset() -> tuple[ObsType, dict]:
        ...
```

The semantics of the interface is the following:
* The environment keeps track of the **current state** with an **internal attribute** `state` of type `ObsType`.
* The `step` method allows us to implement the **transition matrix** $P(\cdot \mid s, a)$ and the **reward function** $R(s, a)$. In particular, it takes as input an `action` of type `ActType` and returns a tuple containing:
    * The **next state** `next_state` of type `ObsType` and the **reward** `reward` of type `float`, generated according to the internal state `state` and the provided `action`.
    * A boolean flag `terminated` which evaluates to `True` if the `next_state` belongs to the subset of **terminal states** $\mathcal{S}^-$.
    * A boolean flag `truncated` which evaluates to `True` if the number of steps taken in the current episode reaches a **horizon** $H$. <br>_This is useful for practical reason, preventing episodes with too many time steps during training._
    * A dictionary `info` containing auxiliary diagnostic data (which we will simply return as an empty dictionary `{}`).
* The `reset` method allows us to implement the **initial state distribution** $\mu_0$. In particular, it resets the internal state of the environment and returns the new initial state `initial_state` of type `ObsType`, alongside an initial `info` dictionary (again, we simply return `{}`).

_Remark._ The `Env` interface allows modeling the more general _Partially Observable_ MDP (POMDP), where the information provided to the agent is an element of the set of _observations_ $\mathcal{O}$ which, in general, can differ from the set of states $\mathcal{S}$. The class `ObsType` represents an element of the set $\mathcal{O}$. In this notebook, we will consider just the standard MDP case, where $\mathcal{O} = \mathcal{S}$. Thus `ObsType` is the class which represents the states of the MDP.

The `gymnasium` library provides a set of ready to use environment well-known in the RL literature that can be instantiated directly through the name of the desired environment.

```python
env = gym.make('<env-name>', render_mode='<mode>')
```

**📝 Example.** The _cliff walking_ environment models a grid world in which the agent has to learn to reach a goal position from a starting state, avoiding falling off the cliff.

<center>
<img src="../src/rl_basics/imgs/cliff_walking_random.gif"
     style="width: 900px; height: 300px; object-fit: cover; border: 1px solid #ccc;" />
</center>

The environment models the following MDP:
* The **state** space is the set of pairs $\mathcal{S} = \{ 0, \dots, 3 \} \times \{ 0, \dots, 11 \}$, representing the current position of the agent in the $4 \times 12$ grid.
* The **terminal states** $\mathcal{S}^- \subset \mathcal{S}$ consist of the goal state $(3, 11)$ and the cliff states $\{ (3, c) \text{ s.t. } c \in \{1, \dots, 10\} \}$. Stepping into any of these states ends the episode naturally (evaluating `terminated` to `True`).
* The agent can do one of the following **actions**: $\mathcal{A} = \{$ Up, Right, Down, Left $\}$ represented with the numbers from $0$ to $3$ in the given order.
* The **state transition** is deterministic: the agent moves in the direction given by the action, unless there is a wall (grid boundary), in which case the state won't change.
* The **reward** function is shaped to encourage the agent to reach the goal as fast as possible without falling off the cliff. The agent receives a $-1$ reward for standard steps. If it steps onto the cliff, it receives a $-100$ reward.
* The agent **starts** from a **random state** safely above the cliff.
* The **horizon** is set to $H = 50$. If the agent survives for $50$ steps of interaction without reaching a terminal state, the episode is artificially halted (evaluating `truncated` to `True`).

Below, you can find an **implementation** of the environment.

```python
UP    = 0
RIGHT = 1
DOWN  = 2
LEFT  = 3

class CliffWalkingEnv(Env):
    ROWS = 4
    COLS = 12
    CLIFF = {(3, c) for c in range(1, 11)}
    GOAL = (3, 11)

    def reset(self):
        valid_states = [
            (r, c)
            for r in range(self.ROWS)
            for c in range(self.COLS)
            if (r, c) not in self.CLIFF and (r, c) != self.GOAL
        ]
        self.state = valid_states[np.random.randint(len(valid_states))]
        self.interactions = 0
        return self.state, {}

    def step(self, action: int):
        r, c = self.state
        if   action == UP    and r > 0:            self.state = (r - 1, c)
        elif action == RIGHT and c < self.COLS - 1: self.state = (r, c + 1)
        elif action == DOWN  and r < self.ROWS - 1: self.state = (r + 1, c)
        elif action == LEFT  and c > 0:             self.state = (r, c - 1)

        self.interactions += 1

        terminated = self.state == self.GOAL
        truncated  = self.interactions >= 50
        reward     = -100 if self.state in self.CLIFF else -1

        if reward == -100:
            terminated = True

        return self.state, reward, terminated, truncated, {}
```

Let's **instantiate** the cliff environment.

In [3]:
CLIFF_WALKING_ENV_NAME = 'CliffWalking-RLSS-v0'
env = gym.make(CLIFF_WALKING_ENV_NAME, render_mode='rgb_array')

### Let's construct our environment!

**🚕 A Normal Day in Milan.** It is a normal day in Milan, and a taxi driver has been tasked with transporting RLSS students from the Politecnico campus to some of the city's most famous landmarks. Our goal is to help the driver navigate the busy urban environment and efficiently bring the students to destinations such as Parco Sempione, the Duomo, and other iconic locations across the city.

Let's **model** this problem with an environment! In particular:

* The **state** is a tuple `(row, col, pass_idx, dest_idx)`:
    * `(row, col)` is the taxi's position in the $5 \times 5$ grid world, with `row, col` $\in \{ 0, \dots, 4 \}$.
    * Four cells of the grid are city landmarks (Parco Sempione, Duomo, etc.), stored in the list `LOCS`.
    * `pass_idx` $\in \{ 0, \dots, 4 \}$ encodes where the passenger is. If `pass_idx = 4` (i.e. `PASS_IN_TAXI`), the passenger is _inside_ the taxi; otherwise they are waiting at cell `LOCS[pass_idx]`.
    * `dest_idx` $\in \{ 0, \dots, 3 \}$ identifies the destination: `LOCS[dest_idx]` is the cell where the passenger wants to arrive.

* The agent chooses one of six **actions**: $\mathcal{A} = \{$ `SOUTH`, `NORTH`, `EAST`, `WEST`, `PICKUP`, `DROPOFF` $\}$, encoded as the integers $0$ to $5$ in that order.
    * Actions $0$–$3$ move the taxi one cell in the grid.
    * **Pick up** (action $4$) loads the passenger, but only when the taxi is in the passenger's cell.
    * **Drop off** (action $5$) unloads the passenger, but only when they are on board and the taxi has reached the destination cell.

* The **state transition** is deterministic:
    * A movement action shifts the taxi one cell in the chosen direction, but the taxi cannot go through walls — e.g., it cannot move from cell `(3, 0)` to `(3, 1)`, whereas moving from `(2, 0)` to `(2, 1)` is allowed.
    * **Pick up**: if executed in the passenger's cell, the passenger boards and `pass_idx` is set to `PASS_IN_TAXI`; otherwise nothing changes.
    * **Drop off**: if the passenger is on board *and* the taxi is at the destination `LOCS[dest_idx]`, the passenger gets off at the destination; otherwise nothing changes. After a successful drop-off, a new passenger spawns at one of the $4$ landmarks chosen at random, with a destination drawn from the $3$ remaining landmarks, and the taxi stays where it is.

* The **reward** is $-1$ at every step, to push the taxi toward delivering passengers as quickly as possible. A successful drop-off at the desired destination gives $+20$. Illegal **Pick up** or **Drop off** actions are penalized with $-10$.

* The **initial state** is sampled uniformly at random: the taxi's starting cell over the whole grid, and the passenger's location and destination over the landmarks in `LOCS` — with the constraint that the destination must differ from the passenger's starting landmark.

* The **horizon** is set to $H = 100$.

<center>
<img src="../src/rl_basics/imgs/custom_taxi_grid.png" width=600 />
</center>

**⭐ Exercise.** Complete the implementation of the `MilanTaxiEnv` according to the above description. Check the image to better understand the grid world.
<br> In particular, you have to implement:
- Part of the `_reset` method, sampling the initial position of the taxi uniformly at random and spawning the passenger.
- The `_transitions` method, handling the movement of the taxi (making sure that it can't go through walls) and the "Pick up" and "Drop off" actions.

_Hint._ You can use the helper function `check_wall` which evaluates to `True` if, by following the usual action transition, the taxi would cross a wall.
<br> The helper method `_spawn_new_passenger` handles the spawn of passengers and the corresponding desired destination.

In [8]:
NUM_ROWS, NUM_COLS = 5, 5
HORIZON = 100

SOUTH, NORTH, EAST, WEST, PICKUP, DROPOFF = 0, 1, 2, 3, 4, 5

INTERNAL_WALLS = [
    ((0, 1), (0, 2)), ((1, 1), (1, 2)),
    ((3, 0), (3, 1)), ((4, 0), (4, 1)),
    ((3, 2), (3, 3)), ((4, 2), (4, 3))
]

def check_wall(row, col, new_row, new_col):
    return ((row, col), (new_row, new_col)) in INTERNAL_WALLS or \
        ((new_row, new_col), (row, col)) in INTERNAL_WALLS

LOCS = [(0, 0), (0, 4), (4, 0), (4, 3)]

PASS_IN_TAXI = 4

class MilanTaxiEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.state = None
        self.time_step = 0
        self.lastaction = None
        self._delivered_passengers = 0

    def reset(self):
        self._delivered_passengers = 0
        self.time_step = 0
        self.lastaction = None

        # Your code goes here: -------------------------------------
        taxi_row = random.randrange(NUM_ROWS)
        taxi_col = random.randrange(NUM_COLS)
        pass_idx = random.randrange(len(LOCS))
        dest_idx = random.randrange(len(LOCS))
        
        while dest_idx == pass_idx:
            dest_idx = random.randrange(len(LOCS))
        # ----------------------------------------------------------

        self.state = (taxi_row, taxi_col, pass_idx, dest_idx)

        return self.state, {}

    def step(self, action):
        row, col, pass_idx, dest_idx = self.state

        new_row, new_col, new_pass_idx, new_dest_idx, reward, dest_reached = \
            self._transitions(row, col, pass_idx, dest_idx, action)

        self.lastaction = action

        self.time_step += 1
        truncated = self.time_step >= HORIZON
        
        if dest_reached:
            self._delivered_passengers += 1
            if not truncated:
                new_pass_idx, new_dest_idx = self._spawn_new_passenger()

        self.state = (new_row, new_col, new_pass_idx, new_dest_idx)

        return self.state, reward, False, truncated, {"delivered_passengers": self._delivered_passengers}

    def _spawn_new_passenger(self):
        pickup, dropoff = np.random.choice(4, size=2, replace=False)
        return int(pickup), int(dropoff)

    def _transitions(self, row, col, pass_idx, dest_idx, action):
        new_row, new_col = row, col
        new_pass_idx = pass_idx
        dest_reached = False
        reward = -1

        # Your code goes here: -------------------------------------
        if action == SOUTH:
            new_row = row + 1
        
        elif action == NORTH:
            new_row = row - 1
        
        elif action == EAST:
            new_col = col + 1
        
        elif action == WEST:
            new_col = col - 1
        
        elif action == PICKUP:
            if pass_idx != PASS_IN_TAXI and (row, col) == LOCS[pass_idx]:
                new_pass_idx = PASS_IN_TAXI
            else:
                reward = -10
        
        elif action == DROPOFF:
            if pass_idx == PASS_IN_TAXI and (row, col) == LOCS[dest_idx]:
                reward = 20
                dest_reached = True
            else:
                reward = -10

        if action in (SOUTH, NORTH, EAST, WEST):
            if (
                0 <= new_row < NUM_ROWS
                and 0 <= new_col < NUM_COLS
                and not check_wall(row, col, new_row, new_col)
            ):
                pass
            else:
                new_row, new_col = row, col
        # ----------------------------------------------------------

        return new_row, new_col, new_pass_idx, dest_idx, reward, dest_reached

    def render(self):
        return render_taxi(self.state, self.lastaction)

    def close(self):
        close_taxi_render()

## Policy

A _policy_ describes the behavior of a RL agent inside of an environment.

**Mathematical Formulation.** Given an MDP with states $\mathcal{S}$ and actions $\mathcal{A}$, a (_Markovian_) policy $\pi$ is a map $\pi : \mathcal{S} \rightarrow \Delta(\mathcal{A})$.
<br> We say that the policy is deterministic whenever $\pi(s) = \delta_{a(s)}$ for every $s \in \mathcal{S}$, where $\delta_a$ is the Dirac Delta distribution over the action $a \in \mathcal{A}$.

**Implementation.** We implement policies through the following class.

In [9]:
class Policy(ABC):
    @abstractmethod
    def get_action(self, state):
        pass

The `get_action` method samples an action from the distribution induced by $\pi$ over the set of actions $\mathcal{A}$, given `state`.

**⭐ Exercise.** Let's create a random policy for action spaces $\mathcal{A} = \{ 0, \dots, N_{\mathcal{A}} - 1 \}$ where $N_{\mathcal{A}}$ is the _number of actions_.
<br>Formally, the random policy is defined as $\pi(s) = \text{Unif}(\{ 0, \dots, N_{\mathcal{A}} - 1 \})$ for every $s \in \mathcal{S}$.

In [10]:
class RandomPolicy(Policy):
    def __init__(self, actions_cardinality):
        self.actions_cardinality = actions_cardinality
    
    def get_action(self, state):
        # Your code goes here: -------------------------------------
        return random.randrange(self.actions_cardinality)
        # ----------------------------------------------------------

**⭐ Exercise.** Let's create an heuristic policy to solve the _cliff walking_ environment. <br>_Hint._ You can use the constants for the action mapping `UP = 0`, `RIGHT = 1`, `DOWN = 2`, `LEFT = 3`,  which are already defined.

In [ ]:
class CliffWalkingHeuristicPolicy(Policy):
    def get_action(self, state):
        # Your code goes here: -------------------------------------
        r, c = state

        if r == 3 and c == 0:
            return UP

        elif r == 2 and c < 11:
            return RIGHT

        elif r == 2 and c == 11:
            return DOWN
        # ----------------------------------------------------------

## Interaction

**Trajectories.** Given a policy $\pi$ and an environment, a trajectory is the sequence of _current state_, _action_ taken by the agent, and corresponding observed reward:

<center>

$\tau = (S_0, A_0, R_0, \dots, S_{H-1}, A_{H-1}, R_{H-1}, S_H)$ where $R_h = R(S_h, A_h)$.

</center>

The trajectory is the mathematical object which represents the interaction of the policy with the environment.

_Remark._ When considering more than a trajectory, we use the notation $S_h(\tau)$, $A_h(\tau)$, $R_h(\tau), H(\tau)$ to differentiate the elements of the different trajectories.
We use $\tau_{h:} = (S_h, A_h, R_h, \dots, S_{H-1}, A_{H-1}, R_{H-1}, S_H)$ when considering only the portion of the trajectory which starts from time-step $h$. Observe that $S_0(\tau_{h:}) = S_h(\tau)$, etc..

**⭐ Exercise.** Given a `Policy` and an `Env`, implement the part of the `rollout` function which simulates the policy on the environment and returns a trajectory as three lists: `states`, i.e., the sequence of states of the environment $(S_0, \dots, S_{H-1})$, `actions`, i.e., the sequence of actions taken by the agent $(A_0, \dots, A_{H-1})$, and `rewards`, i.e., the sequence of rewards $(R_0, \dots, R_{H-1})$ observed by the agent.

_Remark._ When `render` is `True`, the function will return in addition the `frames` list for rendering purposes. Don't worry about this aspect, just focus on the interaction between the policy and the environment.

In [ ]:
def rollout(env: gym.Env, policy: Policy, render=True):
    frames =  []

    initial_state, _ = env.reset()

    states  = [ initial_state ]
    actions = []
    rewards = []

    done = False
    while not done:
        if render:
            frames.append(env.render())
        
        # Your code goes here: -------------------------------------
        ...
        # ----------------------------------------------------------
    
    if render:
        return frames, states, actions, rewards
    else:
        return states, actions, rewards

You can use the `animate_frames` function to **render** the behavior of the agent. Let's try it out with the _heuristic policy_ on the _cliff walking_ environment.

In [ ]:
seed_everything(42)

env     = gym.make(CLIFF_WALKING_ENV_NAME, render_mode='rgb_array')
policy  = CliffWalkingHeuristicPolicy()

frames, _, _, _ = rollout(env, policy)

animate_frames(frames)

Let's simulate also the _Milan taxi_ environment.

In [ ]:
seed_everything(42)

env    = MilanTaxiEnv()
policy = RandomPolicy(actions_cardinality=6)

frames, _, _, _ = rollout(env, policy)

animate_frames(frames)

## Learning Objective

**Expected Discounted Cumulative Reward.** The **goal** of a RL is to maximize the _expected discounted cumulative reward_ over the horizon $H$, for a given discount factor $\gamma \in [0, 1]$, defined as:
$$
J_\gamma^{\pi} = \mathbb{E}_{\mu_0, P, \pi} \left[\sum_{h=0}^{+\infty} \gamma^h R(S_h, A_h) \right].
$$
Given, a trajectory $\tau = (S_0, A_0, R_0, \dots, S_{H-1}, A_{H-1}, R_{H-1}, S_H)$, we can define the _expected discount cumulative reward_, in terms of the _discounted return_ of the trajectory:
$$
G_\gamma(\tau) = \sum_{h=0}^{H(\tau)-1} \gamma^h R_h(\tau).
$$
In particular, $J_\gamma^{\pi} = \mathbb{E}_{\mu_0, P, \pi} \left[ G_\gamma(\tau) \right]$.

_Remark._ We follow the convention $0^0 = 1$ when $\gamma = 0$.

**Value Functions.** The objective $J_\gamma^\pi$ scores a policy with a single number, averaged over where the agent happens to start. To reason about *which states are good* and *which actions to take*, we need the same quantity broken down state by state:
* The **value function** defined as:

<center>

$V_\gamma^\pi(s) = \mathbb{E}_{P,\pi}\left[ \sum_{h = 0}^{+\infty} \gamma^h R(S_h, A_h) \mid S_0 = s \right]$ for every $s \in \mathcal{S}$.

</center>

* The **$Q$ function** defined as:

<center>

$Q_\gamma^\pi(s, a) = \mathbb{E}_{P, \pi}\left[ \sum_{h = 0}^{+\infty} \gamma^h R(S_h, A_h) \mid S_0 = s, A_0 = a \right]$ for every $s \in \mathcal{S}$, $a \in \mathcal{A}$.

</center>

In particular, the value function corresponds to the expected discounted cumulative reward when the agent starts from a given state $s \in \mathcal{S}$.
The $Q$ function computes the same quantity, conditioning in addition on the first action $a \in \mathcal{A}$ taken by the agent. These functions are fundamental to understand which are the desirable states and the actions to be taken in each state. The two quantities are linked as follows:
$$
V_\gamma^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a \mid s) Q_\gamma^\pi(s, a),
$$
$$
Q_\gamma^\pi(s, a) = R(s, a) + \gamma \sum_{s' \in \mathcal{S}} P(s' \mid s, a) V_\gamma^\pi(s').
$$

These two identities are the whole of the rest of the course in miniature. `RL01` solves the second one **exactly**, as a linear system, when $P$ is known; `RL02` **estimates** it from sampled trajectories when it is not.

**⭐ Exercise.** The `evaluate_policy` function takes in input an `Env`, a `Polcy`, a number of runs `n_runs` and the value of $\gamma$ (`discount_factor`). It simulates `n_runs` trajectories. Your task is to compute for each trajectory the corresponding discounted return. These are then averaged, obtaining an estimate of $J^\pi_\gamma$.

In [ ]:
def evaluate_policy(env: gym.Env, policy: Policy, n_runs: int = 100, discount_factor: float = 1.0):
    mean_discounted_return = 0.0
    sum_squared_diffs = 0.0

    for i in tqdm(range(n_runs), desc="Evaluating Policy"):
        _, _, rewards = rollout(env, policy, render=False)
        # Your code goes here: -------------------------------------
        discounted_return = 0.0
        ...
        # ----------------------------------------------------------

        # Welford's algorithm to compute running variance 
        diff_from_old_mean = discounted_return - mean_discounted_return
        mean_discounted_return += diff_from_old_mean / (i + 1)
        
        diff_from_new_mean = discounted_return - mean_discounted_return
        sum_squared_diffs += diff_from_old_mean * diff_from_new_mean

    # Compute sample standard deviation (Bessel's correction)
    if n_runs > 1:
        std_dev = (sum_squared_diffs / (n_runs - 1)) ** 0.5
    else:
        std_dev = 0.0

    # Calculate Margin of Error (95% Confidence Level, z = 1.96)
    margin_of_error = 1.96 * (std_dev / np.sqrt(n_runs)) if n_runs > 0 else 0.0
    
    lower_bound = mean_discounted_return - margin_of_error
    upper_bound = mean_discounted_return + margin_of_error

    # Compact display using formal notation
    from IPython.display import display, Markdown
    
    display(Markdown(
        f"$J^\pi_\\gamma \\in [{lower_bound:.2f}, {upper_bound:.2f}]$ w.p. $0.95$"
    ))

    return mean_discounted_return, margin_of_error

Let's **evaluate** the heuristic policy!

In [ ]:
seed_everything(42)

env     = gym.make(CLIFF_WALKING_ENV_NAME, render_mode='rgb_array')
policy  = RandomPolicy(actions_cardinality=4)

_, _ = evaluate_policy(env, policy)

---
## Where next?

We now have an environment, a policy, a way to make them interact, and a way to score the
result. What we do **not** have is a way to find a *good* policy.

There are two routes, and we take them in order:

- **`RL01_Dynamic_Programming`** — if the environment's transition probabilities are known,
  the optimal policy can be computed **exactly**, with linear algebra.
- **`RL02_Prediction_and_Control`** — when the model is unknown, the same quantities must be
  **estimated from sampled experience**.

**Onwards to `RL01_Dynamic_Programming`!**

---

### Source

Adapted from the **Reinforcement Learning Summer School 2026 (RLSS26)**, Milan, Italy — *RL Basics I: Introduction to Reinforcement Learning*.

Original authors (Politecnico di Milano): [Enea Gusmeroli](mailto:enea.gusmeroli@polimi.it) **•** [Cristiano Migali](mailto:cristiano.migali@polimi.it) **•** [Davide Salaorni](mailto:davide.salaorni@polimi.it) **•** [Gianmarco Tedeschi](mailto:gianmarco.tedeschi@polimi.it)

Original materials: [github.com/gianmtedeschi/tutorial-rlss26](https://github.com/gianmtedeschi/tutorial-rlss26)
